In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

# =========================================================
# 1. 경로 설정
# =========================================================

image_paths = [
    r".\Train\Bengin cases\Bengin case (1).jpg",
    r".\2026\폐암 논문\블록 + 블러\실험 완료\256\Train_Block256\Bengin cases\Bengin case (1).jpg",
    r".\2026\폐암 논문\블록 + 블러\실험 완료\512\Train_Block512_BlurSigma256\Bengin cases\Bengin case (1).jpg",
    r".\2026\폐암 논문\블록 + 블러\실험 완료\256\Train_Block256_BlurSigma32\Bengin cases\Bengin case (1).jpg",
]

labels = [
    "(a) Original",
    "(b) B256_S0",
    "(c) B512_S256",
    "(d) B256_S32"
]

# =========================================================
# 2. 폰트 설정
# =========================================================

font_path = (
    r"fonts"
    r"\NanumGothicBold.ttf"
)

font_size = 45

# 라벨이 들어갈 하단 여백
label_area_h = 60

# 이미지 사이 여백
gap = 4

# =========================================================
# 3. 경로 및 폰트 확인
# =========================================================

for path in image_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"이미지를 찾을 수 없습니다:\n{path}"
        )

if not os.path.exists(font_path):
    raise FileNotFoundError(
        f"폰트를 찾을 수 없습니다:\n{font_path}"
    )

# NanumGothic Bold 25pt
font = ImageFont.truetype(
    font_path,
    font_size
)

print("NanumGothic Bold 25pt 로드 완료")


# =========================================================
# 4. 이미지 불러오기
# =========================================================

imgs = [
    Image.open(path).convert("RGB")
    for path in image_paths
]

# 첫 번째 이미지 크기를 기준으로 통일
target_size = imgs[0].size

imgs = [
    img.resize(target_size)
    for img in imgs
]

w, h = target_size

print(f"기준 이미지 크기: {w} x {h}")


# =========================================================
# 5. 이미지 + 라벨 합치기
# =========================================================

def add_label(img, text):

    # 이미지 아래에 라벨 영역 추가
    canvas = Image.new(
        "RGB",
        (w, h + label_area_h),
        "white"
    )

    # 원본 이미지 붙이기
    canvas.paste(
        img,
        (0, 0)
    )

    draw = ImageDraw.Draw(canvas)

    # 글자 크기 측정
    bbox = draw.textbbox(
        (0, 0),
        text,
        font=font
    )

    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    # 가운데 정렬
    x = (w - text_w) // 2

    # 라벨 영역 중앙 정렬
    y = (
        h
        + (label_area_h - text_h) // 2
        - bbox[1]
    )

    # 검정색 Bold 글자
    draw.text(
        (x, y),
        text,
        font=font,
        fill="black"
    )

    return canvas


panels = [
    add_label(img, label)
    for img, label in zip(imgs, labels)
]


# =========================================================
# 6. 2 x 2 합성
# =========================================================

cell_w = w + gap
cell_h = (h + label_area_h) + gap

grid_w = cell_w * 2 - gap
grid_h = cell_h * 2 - gap

grid = Image.new(
    "RGB",
    (grid_w, grid_h),
    "white"
)

positions = [
    (0, 0),
    (cell_w, 0),
    (0, cell_h),
    (cell_w, cell_h)
]

for panel, pos in zip(
    panels,
    positions
):
    grid.paste(
        panel,
        pos
    )


# =========================================================
# 7. 결과 저장
# =========================================================

output_path = (
    r".\2026\폐암 논문"
    r"\블록 + 블러\실험 완료"
    r"\privacy_composite_2x2.png"
)

grid.save(
    output_path,
    quality=100
)

print()
print("========================================")
print("완료!")
print(f"저장 위치:\n{output_path}")
print("========================================")